# Model #

I try to put down a model. I will use the xor-perceptron model as a base and modify that.

first create the lattice, N by N, with Nsol = N**2

I should create a class for the cell object and then the network class is formed by cell objects and when the network is created, each network is associated with a position in the lattice.

Note: for now I'm only considering one possible link between each pair neuron for each direction

In [1]:
import jax
from jax import random as jrd
from jax import numpy as jnp
from jax import debug as jdb
import graph_tool as gt

In [2]:
# parameters of the simulation
par = {'key': jrd.key(1634),    # key for random generation
       'N': 10,
       'int_range': 1,             # interaction range
       'p_self_link': 0.5,
       'target_set': [0.0,1.0,1.0,0.0],
       'input_set':[[0,0],[0,1],[1,0],[1,1]]}

In [ ]:
# define a quick function for generating a new random key from par['key'] and update par['key']
def gen_key(par):
    key, subkey = jrd.split(par['key'])     # generate a random key by splitting the key in par
    par['key'] = key                        # update the key in par
    return subkey, par                      # I have to return also par, otherwise, bc of JIT's rules, par['key'] won't update

# quick function to calculate the Eucledian distance between two sites in the lattice
def cdist(a,b,vmap=False):
    if vmap:
        fun = jax.vmap(lambda a,b: jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2),in_axes=0)
        return fun(a,b)
    return jnp.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

class Cell(object):
    
    def __init__(self):     
        self.activation = None     # define the activation function through Kolmogorov-Arnold decomposition or Fourier's    # generate a key for random generation 
        self.value = -1    # For now i initialize it to -1, bc I know that can't be 
            
    def generate(self, par):
        self.activation = jax.nn.sigmoid  # DEFINE THIS!!!!
        subkey, par = gen_key(par)                  # generate the subkey for the random generation in the following line
        self.value = jrd.choice(subkey, jnp.arange(2)) # Assign an initial value of either 0 or 1   
    
    def compute(self, input):      # compute the value of the cell given the input and the activation. This will be the value sent to the (eventual) other cells.
        self.value = self.activation(jnp.sum(input))
        

class Network(object):
    
    def __init__(self):             # NON SO SE HA SENSO INIZIALIZZARE COSì LE MATRICI, PERCHè ESSENDO N = 0, VIENE MATRICE = []
        self.N = 0                                                  # N - side of the lattice 
        self.lattice = []                                           # lattice
        self.J = None                                               # weight matrix 
        self.C = None                                               # connectivity matrix 
        self.B = None                                               # bias matrix 
        self.D = None                                               # distance matrix - this is computed and stored for faster execution
        self.graph = None                                           # graph
        self.fitness = None                                         # Initialize it to None for avoiding recomputation 
        
    @property                                                       # I have to make it a property so that it changes dynamically with self.N
    def N_sol(self):
        return self.N **2                                           # N_sol = N**2 - Number of cells in the lattice (each site in the lattice has one cell in it)
    '''
    @property
    def lattice(self):                                              # lattice: N_sol - it's a string in which each spot S is a site in the actual 2D lattice (S = i * N + j)
        self._lattice = [None] * self.N_sol                         # I have to use list bc of JAX. this creates an N_sol list
        return self._lattice
    @lattice.setter
    def lattice(self,value):
        self._lattice = value
    '''
    @property
    def J(self):                                                    
        if self._J is None:
            self._J = jnp.ones((self.N_sol, self.N_sol))
        return self._J
    @J.setter
    def J(self, value):
        self._J = value
    @property
    def C(self):                                                    
        if self._C is None:
            self._C = jnp.ones((self.N_sol, self.N_sol))
        return self._C
    @C.setter
    def C(self, value):
        self._C = value
    @property
    def B(self):                                                    
        if self._B is None:
            self._B = jnp.zeros((self.N_sol, self.N_sol))
        return self._B
    @B.setter
    def B(self, value):
        self._B = value
    @property
    def D(self):                                                    
        if self._D is None:
            self._D = jnp.zeros((self.N_sol, self.N_sol))
        return self._D
    @D.setter
    def D(self, value):
        self._D = value
    @property
    def graph(self):        # Add a graph-tool object as a property for using graph-tool's methods on the network
        if self._graph is None:
            self._graph = gt.Graph(jnp.column_stack(jnp.nonzero(self.C)))
        return self._graph
    @graph.setter
    def graph(self,value):
        self._graph = value

    
    def generate(self,par):
        # First set the parameters of the Network from par
        self.N = par['N']
        # First generate a cell for each lattice site and assign the former to the latter
        for s in range(self.N_sol):
            cell = Cell()                                       # Initialize a Cell object
            cell.generate(par)                                  # Generate it - (also par['key] gets updated here)
            cell.S = s                                          # Assign a new attribute 'S' which is the site in the 1D lattice string
            cell.coord = (s // self.N, s % self.N)              # Assign a new attribute 'coord' to the cell object and set it to the coordinates of the lattice site
            self.lattice.append(cell)                         # Assign the generated cell to the lattice site
        jdb.print('Lattice ready.')
        # Then randomly generate weight, bias and connectivity matrices
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        #jdb.print('{s}',s=self.J.shape)
        self.J = jrd.uniform(subkey,shape=self.J.shape)                 # uniformly populate the weights in the weight matrix J 
        #jdb.print('{j}',j=self.J)     
        subkey, par = gen_key(par)                                      # generate the subkey for the random generation in the following line
        self.B = jrd.normal(subkey,shape=self.B.shape)                  # extract from a normal distribution centered in 0 with st. dv. = 1 the biases (spero vada bene fatto così)
        #self.D = cdist(jnp.array([cell.coord for cell in self.lattice]),       # DA CONTROLLARE!!
        #              jnp.array([cell.coord for cell in self.lattice]),vmap=True) # compute the distance between each two cells, given their coordinates
        self.D = jnp.array([cdist(self.lattice[c].coord,self.lattice[d].coord,vmap=False) 
                            for c in range(self.N_sol) 
                            for d in range(self.N_sol)]).reshape((self.N_sol,self.N_sol))
        subkey, par = gen_key(par)
        prob_matrix = jnp.where(jnp.eye(self.N_sol, dtype=bool),        # define a matrix for the link probability for C
                    par['p_self_link'], 1 / self.D)
        self.C = jnp.where(jrd.bernoulli(subkey, prob_matrix),1,0)      
        jdb.print('Matrices ready.')
        # Already compute the fitness of the network
        self.compute_fitness(par)
    
    # QUI LA FITNESS è PARI PARI A QUELLA DELLO XOR, LA DEVO RIDEFINIRE
    def compute_fitness(self, par):
        if self.fitness == None:        # I need this check, bc otherwise I risk adding fitness over fitness
            self.fitness = 0.           # This is to avoid type conflict and to make sure that I'm not computing the fitness of a network that already has it
            for i,input in enumerate(par['input_set']):
                output = self.ff(input)
                squared_dist = (par['target_set'][i] - output)**2     # square distance between network output and target (theoretical) output
                cost = (jnp.sum(self.C.flatten()))      # here the cost is computed only on the presence or absence of links
                self.fitness += squared_dist + cost # add to the fitness value of the network
            self.fitness /= len(par['input_set'])    # normalize over the inputs
    
    def ff(self, input:list, verb:int=0):
        # set the value of the two inputs cells through the input value
        self.lattice[0].value = input[0]                        # cell in the upper left corner of the 2D lattice
        self.lattice[(self.N - 1) * self.N-1].value = input[1]    # cell in the lower left corner of the 2D lattice
        # Precompute the contributions for each cell to optimize the loop (this is incredibly faster)
        contributions = jnp.dot(self.C * self.J, jnp.array([cell.value for cell in self.lattice])) + jnp.sum(self.B, axis=1)      # weight x value + bias
        for s in range(self.N_sol):
            cell = self.lattice[s]                      # note: here we're treating self links as any other link
            inbound = contributions[s]     # value to give in input to the cell, which then computes its value through the activation
            cell.compute(inbound)           # pass all the contributions through the activation function to determine the value of the cell
        if verb > 0: 
            return [c.value for c in self.lattice]
        output = self.lattice[self.N_sol-1].value         # the cell in the right lower corner is the output
        return output
            

In [ ]:
n = Network()
par['N'] = 2
n.generate(par)
print(n.C)
print(jnp.column_stack(jnp.nonzero(n.C)))
print(jnp.column_stack(jnp.nonzero(n.C)).shape)
g = gt.Graph(jnp.column_stack(jnp.nonzero(n.C)))
g


Lattice ready.
Matrices ready.
[[1 1 1 1]
 [1 0 1 1]
 [1 1 0 1]
 [1 1 1 0]]
[[0 0]
 [0 1]
 [0 2]
 [0 3]
 [1 0]
 [1 2]
 [1 3]
 [2 0]
 [2 1]
 [2 3]
 [3 0]
 [3 1]
 [3 2]]
(13, 2)


<Graph object, directed, with 4 vertices and 13 edges, at 0x7f5ef9fc6810>

In [4]:
# Execution
# Notes: put N as a static argname in .jit()
# REMEMBER TO DEAL WITH THE BOUNDARY CONDITIONS ON THE LATTICE